# Module 4 - Week 6, Topic 5: Model Evaluation Metrics
## Live Demo Notebook

**Scenario:** Opay Nigeria wants to detect fraudulent transactions.
Only ~4% of transactions are fraud - a highly imbalanced dataset.

We expose the **accuracy trap** and learn the metrics that actually matter:
precision, recall, F1, confusion matrix, ROC curve, and AUC.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    roc_curve, auc, roc_auc_score
)

sns.set_theme(style='whitegrid', palette='muted')

df = pd.read_csv('opay_transactions.csv')
print('Shape:', df.shape)
print(f'Fraud rate: {df["is_fraud"].mean():.1%}')
print(f'Fraud count: {df["is_fraud"].sum()} / {len(df)}')
df.head()

In [ ]:
FEATURES = ['amount', 'time_of_day', 'device_type', 'location_match', 'prev_fraud_flag']
TARGET   = 'is_fraud'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Test fraud cases: {y_test.sum()}')

---
## Part 1 - The Accuracy Trap

In [ ]:
# The "dummy" model that always predicts Not Fraud
dummy_preds = np.zeros(len(y_test), dtype=int)  # always predict 0

dummy_accuracy = accuracy_score(y_test, dummy_preds)

print('=== Dummy Model: Always Predict Not Fraud ===')
print(f'Accuracy:  {dummy_accuracy:.2%}  ← looks impressive!')
print(f'Fraud caught: {(dummy_preds[y_test == 1] == 1).sum()} / {y_test.sum()}')
print()
print('The dummy model has ~95% accuracy but catches ZERO fraudulent transactions.')
print('Accuracy is useless as a metric when classes are imbalanced.')

---
## Part 2 - Train a Real Model and Compute All Metrics

In [ ]:
# Train Random Forest
rf = RandomForestClassifier(
    n_estimators=100, max_depth=10,
    class_weight='balanced',   # handle class imbalance
    random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]  # fraud probability

print('=== Random Forest Metrics ===')
print(f'Accuracy:  {accuracy_score(y_test, y_pred):.2%}  (compare to dummy: {dummy_accuracy:.2%})')
print(f'Precision: {precision_score(y_test, y_pred, zero_division=0):.2%}')
print(f'Recall:    {recall_score(y_test, y_pred, zero_division=0):.2%}')
print(f'F1 Score:  {f1_score(y_test, y_pred, zero_division=0):.2%}')
print(f'AUC:       {roc_auc_score(y_test, y_proba):.3f}')

In [ ]:
print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Not Fraud', 'Fraud']))
print()
print('KEY: Look at the "Fraud" row - that is the actual performance measure.')
print('The "accuracy" row is dominated by the majority class (Not Fraud).')

---
## Part 3 - Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Pred: Not Fraud', 'Pred: Fraud'],
    yticklabels=['Actual: Not Fraud', 'Actual: Fraud'],
    ax=ax, linewidths=0.5,
    annot_kws={'size': 14}
)
ax.set_title('Confusion Matrix - Opay Fraud Detection',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'True Negatives  (legit, cleared correctly):   {tn}')
print(f'False Positives (legit, flagged as fraud):     {fp}  ← customer frustrated')
print(f'False Negatives (fraud, missed):               {fn}  ← money lost')
print(f'True Positives  (fraud, caught correctly):     {tp}')
print()
print(f'Fraud caught: {tp}/{tp+fn} = {tp/(tp+fn)*100:.0f}% (Recall)')
print(f'False alarm rate: {fp}/{fp+tn} = {fp/(fp+tn)*100:.1f}% (FPR)')

---
## Part 4 - Precision-Recall Tradeoff

In [ ]:
thresholds = np.arange(0.1, 0.95, 0.05)
precisions, recalls, f1s = [], [], []

for t in thresholds:
    preds = (y_proba >= t).astype(int)
    p = precision_score(y_test, preds, zero_division=0)
    r = recall_score(y_test, preds, zero_division=0)
    f = f1_score(y_test, preds, zero_division=0)
    precisions.append(p)
    recalls.append(r)
    f1s.append(f)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(thresholds, precisions, label='Precision', color='steelblue',  linewidth=2)
ax.plot(thresholds, recalls,    label='Recall',    color='coral',      linewidth=2)
ax.plot(thresholds, f1s,        label='F1 Score',  color='seagreen',   linewidth=2, linestyle='--')
ax.axvline(0.5, color='grey', linestyle=':', label='Default threshold (0.5)')
ax.set_xlabel('Decision Threshold', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Precision vs Recall vs F1 Across Thresholds\n(Lower threshold → catch more fraud but more false alarms)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

best_f1_idx = np.argmax(f1s)
print(f'Best F1 at threshold {thresholds[best_f1_idx]:.2f}: F1={f1s[best_f1_idx]:.3f}')
print(f'  Precision: {precisions[best_f1_idx]:.3f}  Recall: {recalls[best_f1_idx]:.3f}')

---
## Part 5 - ROC Curve and AUC

In [ ]:
# Compute ROC curve
fpr, tpr, thresh = roc_curve(y_test, y_proba)
auc_score = auc(fpr, tpr)

# Also compute for a Logistic Regression for comparison
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000))
])
lr_pipe.fit(X_train, y_train)
lr_proba = lr_pipe.predict_proba(X_test)[:, 1]
lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_proba)
lr_auc = auc(lr_fpr, lr_tpr)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr,    tpr,    color='steelblue', linewidth=2.5,
        label=f'Random Forest (AUC = {auc_score:.3f})')
ax.plot(lr_fpr, lr_tpr, color='coral',     linewidth=2.5, linestyle='--',
        label=f'Logistic Regression (AUC = {lr_auc:.3f})')
ax.plot([0, 1], [0, 1], color='grey', linestyle=':', label='Random classifier (AUC = 0.5)')

ax.fill_between(fpr, tpr, alpha=0.1, color='steelblue')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax.set_title('ROC Curve - Opay Fraud Detection\n(Higher and further left = better)',
             fontsize=12, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Random Forest AUC: {auc_score:.3f}')
print(f'Logistic Reg AUC:  {lr_auc:.3f}')
print(f'Dummy model AUC:   0.500')
print()
print('AUC > 0.8: good separation between fraud and legitimate.')

---
## Part 6 - Full Model Comparison Dashboard

In [ ]:
# Compare Dummy, Logistic Regression, and Random Forest
lr_pred = lr_pipe.predict(X_test)

def get_metrics(y_true, y_pred, y_proba):
    return {
        'Accuracy':  accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall':    recall_score(y_true, y_pred, zero_division=0),
        'F1':        f1_score(y_true, y_pred, zero_division=0),
        'AUC':       roc_auc_score(y_true, y_proba),
    }

results = {
    'Dummy (all Not Fraud)': get_metrics(y_test, dummy_preds, np.zeros(len(y_test))),
    'Logistic Regression':   get_metrics(y_test, lr_pred,    lr_proba),
    'Random Forest':         get_metrics(y_test, y_pred,     y_proba),
}

metrics_df = pd.DataFrame(results).T

print('=== Model Comparison Dashboard ===')
print(metrics_df.round(3).to_string())
print()
print('KEY INSIGHT:')
print('  Dummy model: 95%+ accuracy, but 0 AUC and 0 recall for fraud.')
print('  The Real models have lower accuracy but actually catch fraud.')
print()
print('For fraud detection: Recall and AUC are the metrics that matter.')

---
## Summary

| Metric | What it measures | When it matters |
|---|---|---|
| Accuracy | % correct overall | Only for balanced classes |
| Precision | When predicting fraud, how often correct? | When FP is costly |
| Recall | Of all actual fraud, how much caught? | When FN is costly |
| F1 | Harmonic mean of precision and recall | When both matter |
| AUC | Class separation at all thresholds | Model comparison |

**Golden rule:** On imbalanced data, report metrics per class. The weighted average misleads.

**Next - Topic 6:** Cross-Validation & Hyperparameter Tuning. A single train/test split can be misleading. We learn how to get stable performance estimates and systematically find the best model configuration.